In [1]:
from pyspark.sql.functions import to_timestamp, col

# Load orders with explicit type casting
df_orders_bronze = spark.read.option("header", "true").csv("Files/bronze/olist/olist_orders_dataset.csv")

df_orders_silver = (
    df_orders_bronze
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at")))
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date")))
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))
)

df_orders_silver.printSchema()
print("Orders row count:", df_orders_silver.count())

StatementMeta(, dbb33c3c-4e9e-41be-b465-ec1a98d1dcd2, 3, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Orders row count: 99441


In [2]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

df_reviews_bronze = spark.read.option("header", "true").csv("Files/bronze/olist/olist_order_reviews_dataset.csv")

# Cast timestamp columns
df_reviews_typed = (
    df_reviews_bronze
    .withColumn("review_creation_date", to_timestamp(col("review_creation_date")))
    .withColumn("review_answer_timestamp", to_timestamp(col("review_answer_timestamp")))
)

# Dedupe: keep the latest row per review_id
window_spec = Window.partitionBy("review_id").orderBy(desc("review_answer_timestamp"))
df_reviews_deduped = (
    df_reviews_typed
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("After dedup:", df_reviews_deduped.count())

# Drop orphaned reviews (inner join to orders)
df_reviews_silver = df_reviews_deduped.join(
    df_orders_silver.select("order_id"), "order_id", "inner"
)

print("After dropping orphans:", df_reviews_silver.count())

StatementMeta(, dbb33c3c-4e9e-41be-b465-ec1a98d1dcd2, 4, Finished, Available, Finished, False)

After dedup: 102958
After dropping orphans: 98410


In [3]:
df_orders_silver.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
df_reviews_silver.write.format("delta").mode("overwrite").saveAsTable("silver_reviews")

print("Silver tables written successfully.")

StatementMeta(, dbb33c3c-4e9e-41be-b465-ec1a98d1dcd2, 5, Finished, Available, Finished, False)

Silver tables written successfully.


**Quick sanity check**

In [1]:
df_check_orders = spark.sql("SELECT COUNT(*) as cnt FROM silver_orders")
df_check_reviews = spark.sql("SELECT COUNT(*) as cnt FROM silver_reviews")
df_check_orders.show()
df_check_reviews.show()

StatementMeta(, dc654304-df6d-4c3b-a8c6-76bd0f5c5ce5, 4, Finished, Available, Finished, False)

+-----+
|  cnt|
+-----+
|99441|
+-----+

+-----+
|  cnt|
+-----+
|98410|
+-----+



**Confirm the timestamp casting implementaion**

In [2]:
spark.sql("DESCRIBE silver_orders").show(20, truncate=False)

StatementMeta(, dc654304-df6d-4c3b-a8c6-76bd0f5c5ce5, 7, Finished, Available, Finished, False)

+-----------------------------+---------+-------+
|col_name                     |data_type|comment|
+-----------------------------+---------+-------+
|order_id                     |string   |NULL   |
|customer_id                  |string   |NULL   |
|order_status                 |string   |NULL   |
|order_purchase_timestamp     |timestamp|NULL   |
|order_approved_at            |timestamp|NULL   |
|order_delivered_carrier_date |timestamp|NULL   |
|order_delivered_customer_date|timestamp|NULL   |
|order_estimated_delivery_date|timestamp|NULL   |
+-----------------------------+---------+-------+



**Confirm no duplicate review_ids survived**


In [3]:
spark.sql("""
    SELECT review_id, COUNT(*) as cnt
    FROM silver_reviews
    GROUP BY review_id
    HAVING COUNT(*) > 1
""").show()

StatementMeta(, dc654304-df6d-4c3b-a8c6-76bd0f5c5ce5, 8, Finished, Available, Finished, False)

+---------+---+
|review_id|cnt|
+---------+---+
+---------+---+



**Confirm no orphaned order_ids survived**

In [4]:
spark.sql("""
    SELECT r.order_id
    FROM silver_reviews r
    LEFT ANTI JOIN silver_orders o ON r.order_id = o.order_id
""").show()

StatementMeta(, dc654304-df6d-4c3b-a8c6-76bd0f5c5ce5, 9, Finished, Available, Finished, False)

+--------+
|order_id|
+--------+
+--------+

